In [ ]:
import os
# conda의 numpy(MKL)와 pip의 torch(easyocr)가 OpenMP DLL(libiomp5md.dll)을 각각 불러와 충돌 → 커널이 죽는 문제 방지
# 반드시 numpy / torch / easyocr 를 import 하기 전에 설정해야 함
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import csv
import io
import re
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from PIL import Image

BASE_URL = "https://www.xn--9t4ba1o42k80b431a.com/default/"
OUTPUT_CSV = "eos_main_scrape.csv"
MIN_IMG_SIZE = 80          # 이 크기(px)보다 작은 이미지(아이콘 등)는 OCR 생략
INCLUDE_PDF = True         # 카탈로그 PDF 텍스트 추출 여부
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; text-scraper/1.0)"}

# --- OCR 엔진은 필요할 때만 로딩 (처음 로딩에 시간이 걸림)
_reader = None
def get_reader():
    global _reader
    if _reader is None:
        import easyocr
        _reader = easyocr.Reader(["ko", "en"], gpu=False)
    return _reader


def fetch(url):
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r


def extract_html_text(main):
    """main 영역의 일반 텍스트를 줄 단위로 추출 (중복 제거, 순서 유지)"""
    for tag in main(["script", "style", "noscript"]):
        tag.decompose()
    rows, seen = [], set()
    for line in main.get_text(separator="\n").splitlines():
        line = re.sub(r"\s+", " ", line).strip()
        if line and line not in seen:
            seen.add(line)
            rows.append(line)
    return rows


def extract_image_urls(main):
    """img 태그(src, data-src, srcset)와 inline style의 background-image 수집"""
    urls = []

    def add(u):
        if not u or u.startswith("data:"):
            return
        full = urljoin(BASE_URL, u.strip())
        if full.lower().split("?")[0].endswith(".svg"):
            return
        if full not in urls:
            urls.append(full)

    for img in main.find_all("img"):
        for attr in ("src", "data-src", "data-original", "data-lazy"):
            add(img.get(attr))
        srcset = img.get("srcset")
        if srcset:
            add(srcset.split(",")[0].strip().split(" ")[0])

    for el in main.find_all(style=True):
        for m in re.findall(r"url\(['\"]?([^'\")]+)['\"]?\)", el["style"]):
            add(m)

    return urls


def ocr_image(url):
    """이미지를 내려받아 OCR 수행. (텍스트, 메모) 반환"""
    try:
        data = fetch(url).content
        img = Image.open(io.BytesIO(data))
        if img.width < MIN_IMG_SIZE or img.height < MIN_IMG_SIZE:
            return "", "skipped: too small"
        img = img.convert("RGB")
        # 작은 글자 인식률을 높이기 위해 너무 작으면 확대
        if img.width < 1000:
            ratio = 1000 / img.width
            img = img.resize((int(img.width * ratio), int(img.height * ratio)))
        import numpy as np
        results = get_reader().readtext(np.array(img), detail=1, paragraph=False)
        # 신뢰도 0.3 이상만 사용
        texts = [t for (_, t, conf) in results if conf >= 0.3]
        return " ".join(texts).strip(), f"{len(texts)} boxes"
    except Exception as e:
        return "", f"error: {e}"


def extract_pdf_text(url):
    import fitz  # PyMuPDF
    rows = []
    try:
        doc = fitz.open(stream=fetch(url).content, filetype="pdf")
        for i, page in enumerate(doc, 1):
            text = page.get_text().strip()
            rows.append((i, text if text else "", "" if text else "no text layer (OCR 필요)"))
    except Exception as e:
        rows.append((0, "", f"error: {e}"))
    return rows


def main():
    resp = fetch(BASE_URL)
    # 이 사이트는 EUC-KR 인코딩인데 서버 헤더에 표시가 없어 requests가 ISO-8859-1로 잘못 해석 → 한글 깨짐 방지
    resp.encoding = resp.apparent_encoding
    soup = BeautifulSoup(resp.text, "lxml")
    main_el = soup.find("main") or soup.body

    # 링크 중 PDF 찾기 (main을 정리하기 전에)
    pdf_urls = [
        urljoin(BASE_URL, a["href"])
        for a in main_el.find_all("a", href=True)
        if a["href"].lower().split("?")[0].endswith(".pdf")
    ]
    image_urls = extract_image_urls(main_el)
    text_lines = extract_html_text(main_el)

    rows = []  # (type, source, order, content, note)

    for i, line in enumerate(text_lines, 1):
        rows.append(("html_text", BASE_URL, i, line, ""))

    print(f"HTML 텍스트 {len(text_lines)}줄, 이미지 {len(image_urls)}개, PDF {len(pdf_urls)}개 발견")

    for i, url in enumerate(image_urls, 1):
        print(f"[OCR {i}/{len(image_urls)}] {url}")
        text, note = ocr_image(url)
        rows.append(("image_ocr", url, i, text, note))

    if INCLUDE_PDF:
        for pdf in pdf_urls:
            print(f"[PDF] {pdf}")
            for page_no, text, note in extract_pdf_text(pdf):
                rows.append(("pdf_text", pdf, page_no, text, note))

    # utf-8-sig: 엑셀에서 한글이 깨지지 않도록
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(["type", "source", "order", "content", "note"])
        w.writerows(rows)

    print(f"완료: {OUTPUT_CSV} ({len(rows)}행)")


if __name__ == "__main__":
    main()